In [1]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import os, numpy as np

load_dotenv("env.env")
API_KEY = os.getenv("API_KEY")
API_BASE = os.getenv("API_BASE")

d:\Miniconda\envs\aiagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
y = 10
def f(x: int) -> int:
    # global y
    y = 3
    return x * y
print(f(2))
print(y)

6
10


In [3]:
# 存储文档变量的全局变量

document_content = ""

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [4]:
@tool
def update(content:str) -> str:
    """Updates the documents with the provided content."""
    global document_content
    document_content = content
    return f"Updated, current document content: \n{document_content}"

@tool
def save(filename:str) -> str:
    """Saves the current document content to a text file and finish the process.
    
    Args:
        filename: Name for the text file.
    """

    global document_content

    if not filename.endswith(".txt"):
        filename = f"{filename}.txt"

    try:
        with open(filename, "w") as file:
            file.write(document_content)
        print(f"Document content saved to {filename}.")
        return f"Document content saved to {filename}. Process finished."
    except Exception as e:
        return f"Error saving document: {str(e)}"

tools = [update, save]

In [5]:
model = ChatOpenAI(model="qwen-plus", api_key=API_KEY, base_url=API_BASE).bind_tools(tools) 

def the_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content=f"""
    Your are a Drafter, a helpful writing assistant. You are going to help the user update and save documents. You have two tools:
        - If the user wants to update the document, use the update tool, and provide the content to be updated.
        - If the user wants to save the document, use the save tool, and provide the filename for the text file.
    
    The current document content is:{document_content}                              
                                  """)
    
    if not state["messages"]:
        user_input = "I'm ready to help you update a document. What would you like to create"
        user_message = HumanMessage(content=user_input)

    else:
        user_input = input("\nWhat would like to do with the document?")
        print(f"USER:{user_input}")
        user_message = HumanMessage(content=user_input)

    all_message = [system_prompt] + list(state["messages"]) + [user_message]

    response = model.invoke(all_message)

    print(f"\nAI:{response.content}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

    return {"messages": list(state["messages"]) + [user_message, response]}

def should_coutinue(state: AgentState) -> str:
    messages = state["messages"]

    if not messages:
        return "continue"
    
    for message in reversed(messages):
        if (isinstance(message, ToolMessage) and 
            "saved" in message.content.lower() and 
            "document" in message.content.lower()):
            return "end"
        
    return "continue"

def print_messages(message):
    if not message:
        return
    
    for message in message[-3:]:
        if isinstance(message, ToolMessage):
            print(f"TOOL RESULT: {message.content}")

In [6]:
graph = StateGraph(AgentState)

graph.add_node("the_agent", the_agent)
graph.add_node("tools", ToolNode(tools))

graph.set_entry_point("the_agent")

graph.add_edge("the_agent", "tools")

graph.add_conditional_edges(
    "tools",
    should_coutinue,
    {
        "continue": "the_agent",
        "end": END
    }
)

app = graph.compile()

In [7]:
def run_document_agent():
    print("\n ====== DRAFTER AGENT STARTED ======")

    state = {"messages": []}

    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])

    print("\n ====== DRAFTER AGENT FINISHED ======")

In [ ]:
run_document_agent()


 ====== DRAFTER AGENT STARTED ======

AI:Sure! Could you please let me know:

- What content you'd like to add or update in the document?
- Or, if you have a specific purpose (e.g., a draft email, a meeting note, a story, a report), I can help you write it from scratch.

Once you provide the details, I’ll update the document accordingly.
USER:

AI:It looks like your message may have come through empty. Could you please clarify:

- What content you'd like to add or update in the document?  
- Or, what kind of document you'd like to create (e.g., a letter, a summary, a to-do list, a creative piece)?

I'm ready to help — just let me know!
